# SALES OPTIMIZATION (OLIST)

## Importing Libraries

In [1]:
import pandas as pd
import numpy as np

## Data Loading

In [3]:
orders = pd.read_csv(r"D:\Datasets\Olist Dataset\olist_orders_dataset.csv")
order_items = pd.read_csv(r"D:\Datasets\Olist Dataset\olist_order_items_dataset.csv")
products = pd.read_csv(r"D:\Datasets\Olist Dataset\olist_products_dataset.csv")
customers = pd.read_csv(r"D:\Datasets\Olist Dataset\olist_customers_dataset.csv")
payments = pd.read_csv(r"D:\Datasets\Olist Dataset\olist_order_payments_dataset.csv")
reviews = pd.read_csv(r"D:\Datasets\Olist Dataset\olist_order_reviews_dataset.csv")
print("Data Loaded Successfully")

Data Loaded Successfully


## Basic Cleaning

In [4]:
# Converting date columns
date_cols = ['order_purchase_timestamp','order_approved_at','order_delivered_customer_date','order_estimated_delivery_date']

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

In [6]:
# Drop duplicates
orders.drop_duplicates(inplace=True)
order_items.drop_duplicates(inplace=True)
products.drop_duplicates(inplace=True)
customers.drop_duplicates(inplace=True)
payments.drop_duplicates(inplace=True)
reviews.drop_duplicates(inplace=True)

print("Basic Cleaning Done")

Basic Cleaning Done


## Merging Datasets

In [8]:
df = orders.merge(order_items, on="order_id", how="left") \
           .merge(products, on="product_id", how="left") \
           .merge(customers, on="customer_id", how="left") \
           .merge(payments, on="order_id", how="left") \
           .merge(reviews, on="order_id", how="left")

print("Data Merged Successfully")
print("Shape after merge:", df.shape)

Data Merged Successfully
Shape after merge: (119143, 36)


## Feature Engineering

In [10]:
# Revenue per item
df['revenue'] = df['price'] + df['freight_value']

In [11]:
# Delivery time (in days)
df['delivery_time'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days

In [12]:
# Delay (actual vs estimated)
df['delivery_delay'] = (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.days

In [13]:
# Order value (total per order)
order_value = df.groupby('order_id')['revenue'].sum().reset_index()
order_value.columns = ['order_id', 'total_order_value']

df = df.merge(order_value, on='order_id', how='left')


In [14]:
# Month-Year for trend analysis
df['order_month'] = df['order_purchase_timestamp'].dt.to_period('M')

print("Feature Engineering Done")

Feature Engineering Done


## Handling Missing Values

In [15]:
# Fill delivery-related missing values
df['delivery_time'] = df['delivery_time'].fillna(df['delivery_time'].median())
df['delivery_delay'] = df['delivery_delay'].fillna(0)

In [16]:
# Fill review score (if missing)
df['review_score'] = df['review_score'].fillna(df['review_score'].median())

print("Missing Values Handled")

Missing Values Handled


## Columns To Keep

In [ ]:
columns_to_keep = ['order_id', 'customer_unique_id', 'product_id','product_category_name','customer_state','payment_type','price','freight_value','revenue',
    'total_order_value',
    'review_score',
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'delivery_time',
    'delivery_delay',
    'order_month']

df_final = df[columns_to_keep]

df_final.to_csv("olist_final.csv", index=False)

## Saving Dataset

In [17]:
df.to_csv("olist_cleaned_data.csv", index=False)

print("Cleaned dataset saved as 'olist_cleaned_data.csv'")

Cleaned dataset saved as 'olist_cleaned_data.csv'


In [19]:
print("\nFinal Dataset Info:")
print(df.info())

print("\nSample Data:")
print(df.head(5))


Final Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119143 entries, 0 to 119142
Data columns (total 41 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       119143 non-null  object        
 1   customer_id                    119143 non-null  object        
 2   order_status                   119143 non-null  object        
 3   order_purchase_timestamp       119143 non-null  datetime64[ns]
 4   order_approved_at              118966 non-null  datetime64[ns]
 5   order_delivered_carrier_date   117057 non-null  object        
 6   order_delivered_customer_date  115722 non-null  datetime64[ns]
 7   order_estimated_delivery_date  119143 non-null  datetime64[ns]
 8   order_item_id                  118310 non-null  float64       
 9   product_id                     118310 non-null  object        
 10  seller_id                      118310 non-null 